In [15]:
import os
import json
import numpy as np
import glob

def cargar_matrices_calibracion(ruta_json):
    """
    Carga el archivo calibration.json y extrae las matrices extrínsecas (gTl)
    para los 4 LiDAR de infraestructura.
    """
    with open(ruta_json, 'r') as f:
        calib = json.load(f)
    
    matrices = {
        '11': np.array(calib['crossing1_11_lidar']['extrinsics']['gTl']),
        '12': np.array(calib['crossing1_12_lidar']['extrinsics']['gTl']),
        '31': np.array(calib['crossing1_31_lidar']['extrinsics']['gTl']),
        '32': np.array(calib['crossing1_32_lidar']['extrinsics']['gTl'])
    }
    return matrices

def transformar_nube(ruta_npz, matriz_transformacion):
    """
    Abre un archivo .npz, extrae las coordenadas x, y, z y las multiplica 
    por la matriz de transformación para pasarlas al sistema global.
    """
    # 1. Cargar datos locales del LiDAR
    data = np.load(ruta_npz)
    x = data['x']
    y = data['y']
    z = data['z']
    
    # 2. Crear matriz de coordenadas homogéneas: forma (4, N)
    ones = np.ones_like(x)
    puntos_locales = np.vstack((x, y, z, ones))
    
    # 3. Multiplicación matricial (Global = Matriz @ Local)
    puntos_globales = matriz_transformacion @ puntos_locales
    
    # 4. Devolver (N, 3) descartando la fila de unos
    return puntos_globales[:3, :].T

def fusionar_frame_cruce(directorio_base, timestamp, matrices):
    """
    Busca los 4 archivos correspondientes al mismo instante de tiempo (timestamp)
    en las subcarpetas de cada LiDAR, los transforma y los une.
    """
    nubes_transformadas = []
    
    # Mapeo de sufijos de subcarpetas según la estructura de tus datos
    sensores = {
        '11': 'crossing1_11_lidar',
        '12': 'crossing1_12_lidar',
        '31': 'crossing1_31_lidar',
        '32': 'crossing1_32_lidar'
    }
    
    for sensor_id, subcarpeta in sensores.items():
        # Construir la ruta al archivo específico de este sensor en este timestamp
        ruta_archivo = os.path.join(directorio_base, subcarpeta, f"{timestamp}.npz")
        
        if os.path.exists(ruta_archivo):
            matriz = matrices[sensor_id]
            nube_global = transformar_nube(ruta_archivo, matriz)
            nubes_transformadas.append(nube_global)
        else:
            print(f"Aviso: Falta archivo para el sensor {sensor_id} en el frame {timestamp}")
            
    # Concatenar todas las nubes de este frame en una sola matriz (N_total, 3)
    if nubes_transformadas:
        nube_fusionada = np.vstack(nubes_transformadas)
        return nube_fusionada
    else:
        return np.array([])

In [16]:
import open3d as o3d
import numpy as np

def eliminar_suelo_ransac_o3d(nube_fusionada, distance_threshold=0.3, ransac_n=3, num_iterations=1000):
    """
    Elimina el suelo de una nube de puntos 3D utilizando RANSAC de Open3D.
    
    Parámetros:
    - nube_fusionada: numpy array de forma (N, 3) con las coordenadas [X, Y, Z].
    - distance_threshold: Distancia máxima de un punto al plano para considerarlo suelo (en metros).
    - ransac_n: Puntos necesarios para estimar el plano (3 para un plano 3D).
    - num_iterations: Número de iteraciones del algoritmo.
    """
    # 1. Convertir numpy array a geometría de Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(nube_fusionada)
    
    # 2. Aplicar RANSAC para encontrar el plano del suelo
    # Devuelve los coeficientes del plano [a, b, c, d] y los índices de los puntos que forman el suelo (inliers)
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold,
        ransac_n=ransac_n,
        num_iterations=num_iterations
    )
    
    [a, b, c, d] = plane_model
    print(f"Plano de la carretera detectado: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")
    
    # 3. Extraer los puntos que NO son el suelo (outliers -> vehículos y objetos)
    pcd_sin_suelo = pcd.select_by_index(inliers, invert=True)
    
    # 4. Devolver de nuevo como numpy array
    puntos_sin_suelo = np.asarray(pcd_sin_suelo.points)
    
    return puntos_sin_suelo

In [17]:
import os
import glob
import json
import numpy as np

# 1. Definir la ruta base donde se encuentran las secuencias
ruta_base_crossing = os.path.join("data", "crossing")

# 2. Obtener automáticamente todas las carpetas de secuencias (ignorando __MACOSX)
secuencias = [
    d for d in os.listdir(ruta_base_crossing) 
    if os.path.isdir(os.path.join(ruta_base_crossing, d)) and not d.startswith("_")
]

print(f"Secuencias encontradas automáticamente: {secuencias}")
# Diccionario para almacenar las nubes procesadas (Opcional, según tu memoria RAM)
nubes_por_secuencia = {}

for secuencia in secuencias:
    secuencia_path = os.path.join(ruta_base_crossing, secuencia)
    print(f"\n--- Procesando secuencia: {secuencia} ---")

    # 1. Cargar calibración específica de esta secuencia
    ruta_json = os.path.join(secuencia_path, "calibration.json")
    matrices_calib = cargar_matrices_calibracion(ruta_json)

    # Si por alguna razón la subcarpeta tiene otro nombre exacto en disco, lo listamos para depurar si vuelve a fallar
    if len(archivos_npz) == 0:
        print(f"Aviso: No se encontraron .npz en {ruta_lidar11}. Contenido de la carpeta de secuencia:")
        print(os.listdir(secuencia_path))
    
    # 2. Obtener todos los timestamps sincronizados
    # Como están sincronizados, usamos el LiDAR 11 como referencia para saber qué timestamps existen
    ruta_lidar11 = os.path.join(secuencia_path, "crossing1_11_lidar")
    archivos_npz = glob.glob(os.path.join(ruta_lidar11, "*.npz"))
    
    # Extraer solo el número (timestamp) del nombre del archivo y ordenarlos cronológicamente
    timestamps = sorted([os.path.basename(f).replace('.npz', '') for f in archivos_npz])
    print(f"Encontrados {len(timestamps)} frames sincronizados.")
    
    nubes_fusionadas_secuencia = []
    
    # 3. Fusionar cada frame
    for ts in timestamps:
        nube_completa = fusionar_frame_cruce(secuencia, ts, matrices_calib)
        if len(nube_completa) > 0:
            # Aplicar RANSAC para eliminar el suelo y conservar solo los objetos/vehículos
            nube_limpia = eliminar_suelo_ransac_o3d(nube_completa, distance_threshold=0.3)
        else:
            nube_limpia = np.array([])
        nubes_fusionadas_secuencia.append({
            'timestamp': ts,
            'puntos': nube_completa
        })
        
        # OPCIONAL: Guardar la nube gigante en el disco para no saturar la RAM
        # np.savez_compressed(f"fusions/{secuencia}_{ts}.npz", puntos=nube_completa)
        
    nubes_por_secuencia[secuencia] = nubes_fusionadas_secuencia
    print(f"Secuencia {secuencia} completada.")

Secuencias encontradas automáticamente: ['20241126_0008_crossing1_01', '20241126_0024_crossing1_09', '20241127_0000_crossing1_00']

--- Procesando secuencia: 20241126_0008_crossing1_01 ---
Aviso: No se encontraron .npz en 20241127_0000_crossing1_00\crossing1_11_lidar. Contenido de la carpeta de secuencia:
['.DS_Store', 'calibration.json', 'crossing1_11_lidar', 'crossing1_12_lidar', 'crossing1_31_lidar', 'crossing1_32_lidar', 'timesync_info.csv', 'weather_data.json']
Encontrados 403 frames sincronizados.
Aviso: Falta archivo para el sensor 11 en el frame 1732633536950
Aviso: Falta archivo para el sensor 12 en el frame 1732633536950
Aviso: Falta archivo para el sensor 31 en el frame 1732633536950
Aviso: Falta archivo para el sensor 32 en el frame 1732633536950
Aviso: Falta archivo para el sensor 11 en el frame 1732633537000
Aviso: Falta archivo para el sensor 12 en el frame 1732633537000
Aviso: Falta archivo para el sensor 31 en el frame 1732633537000
Aviso: Falta archivo para el sensor 

In [18]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.colors import to_hex
import numpy as np
import matplotlib.pyplot as plt

def animar_nubes_cruce(nubes_secuencia, num_frames=50, interval_ms=400):
    """
    Crea una animación estilo vídeo 3D a partir de la lista de nubes fusionadas y limpias (sin suelo).
    
    Parámetros:
    - nubes_secuencia: La lista diccionarios con la estructura [{'timestamp': ts, 'puntos': array}, ...]
    """
    # Limitar el número de frames para no saturar la memoria del navegador
    nubes_a_animar = nubes_secuencia[:num_frames]
    N_FRAMES_ANIM = len(nubes_a_animar)
    
    if N_FRAMES_ANIM == 0:
        print("No hay nubes para animar.")
        return

    # Calcular límites FIJOS de los 3 ejes sobre todas las nubes a animar
    # (Evita que la cámara salte y permite apreciar el movimiento real)
    all_points = np.vstack([item['puntos'] for item in nubes_a_animar if len(item['puntos']) > 0])
    
    if len(all_points) == 0:
        print("Todas las nubes están vacías.")
        return

    x_min, x_max = all_points[:, 0].min(), all_points[:, 0].max()
    y_min, y_max = all_points[:, 1].min(), all_points[:, 1].max()
    z_min, z_max = all_points[:, 2].min(), all_points[:, 2].max()

    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111, projection='3d')

    def update(frame_i):
        ax.cla()  
        item = nubes_a_animar[frame_i]
        puntos = item['puntos']
        ts = item['timestamp']
        
        if len(puntos) > 0:
            x = puntos[:, 0]
            y = puntos[:, 1]
            z = puntos[:, 2]
            # Pintamos los puntos de la intersección (por ahora en un tono azulado/gris uniforme sin tracking)
            ax.scatter(x, y, z, s=1, c='#1f77b4', alpha=0.6)
        
        # Configuración fija de la "cámara" del gráfico 3D
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_zlim(z_min, z_max)
        ax.set_box_aspect((x_max - x_min, y_max - y_min, z_max - z_min))
        
        ax.set_xlabel('Global X')
        ax.set_ylabel('Global Y')
        ax.set_zlabel('Global Z')
        ax.set_title(f"Frame {frame_i} — Timestamp: {ts} — Puntos objetos: {len(puntos)}")

    anim = FuncAnimation(fig, update, frames=N_FRAMES_ANIM, interval=interval_ms)
    plt.close(fig)  
    return HTML(anim.to_jshtml())

# --- CÓMO EJECUTARLA ---
# Selecciona una de tus secuencias ya procesadas y pásasela:
sec_a_visualizar = nubes_por_secuencia["20241126_0008_crossing1_01"]
animacion_html = animar_nubes_cruce(sec_a_visualizar, num_frames=50, interval_ms=200)
animacion_html

ValueError: need at least one array to concatenate

In [ ]:
# Comprobación de diagnóstico
nombre_secuencia = "20241126_0008_crossing1_01"

if nombre_secuencia in nubes_por_secuencia:
    secuencia_datos = nubes_por_secuencia[nombre_secuencia]
    print(f"Número total de frames guardados en la secuencia: {len(secuencia_datos)}")
    
    if len(secuencia_datos) > 0:
        # Revisemos el primer frame
        primer_frame = secuencia_datos[0]
        print(f"Timestamp del frame 0: {primer_frame['timestamp']}")
        print(f"Número de puntos de objetos (sin suelo) en el frame 0: {len(primer_frame['puntos'])}")
    else:
        print("La lista de la secuencia está vacía (0 frames).")
else:
    print(f"La clave '{nombre_secuencia}' no existe en el diccionario nubes_por_secuencia.")

Número total de frames guardados en la secuencia: 0
La lista de la secuencia está vacía (0 frames).


In [ ]:
import os
import glob

# Pon aquí la ruta exacta de tu secuencia tal cual la tienes en el explorador
nombre_secuencia = "20241126_0008_crossing1_01"  # o la ruta completa si es necesario

ruta_lidar11 = os.path.join(nombre_secuencia, "crossing1_11_lidar")
print("Buscando archivos en:", ruta_lidar11)
print("¿Existe la carpeta del lidar 11?", os.path.exists(ruta_lidar11))

# Buscar los archivos .npz
archivos_npz = glob.glob(os.path.join(ruta_lidar11, "*.npz"))
print(f"Archivos .npz encontrados en el lidar 11: {len(archivos_npz)}")

if len(archivos_npz) > 0:
    print("Ejemplo de primer archivo encontrado:", archivos_npz[0])

Buscando archivos en: 20241126_0008_crossing1_01\crossing1_11_lidar
¿Existe la carpeta del lidar 11? False
Archivos .npz encontrados en el lidar 11: 0
